# 2D Optimization with Different Template Models

This notebook compares different template parameterization models for 2D membrane segmentation refinement.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.notebook import tqdm

from diffmeshopt.opt2d.optimize import ContourRefiner
from diffmeshopt.opt2d.props import OptimizationProps, SamplingProps, TemplateProps
import diffmeshopt.opt2d.vis as vis2d

In [ ]:
# 1. Load Data
data_path = Path("../data/2d_training_data.pkl")
if not data_path.exists():
    # Fallback to absolute path if running from root
    data_path = Path("/workspace/diffmeshopt/data/2d_training_data.pkl")

print(f"Loading data from {data_path}...")
data = joblib.load(data_path)
image_np = -data["image"]  # Invert intensity
contour_np = data["contour"]
gt_contour_np = data["gt"]

In [ ]:
# 2. Define Props
opt_props = OptimizationProps(
    lr=0.05,
    w_edge=10,
    w_laplacian=50,
    w_sigma_reg=1.0,
    w_template_shape=0.5,
    w_template_smooth=10.0,
)
sampl_props = SamplingProps(num_samples=51, sample_length=1.0)
template_props = TemplateProps(sigma=0.75, peak_dist=4.5, num_samples=51, min_peak_ratio=4.0)

In [ ]:
def run_optimization(template_mode, steps=200):
    print(f"Running optimization with template_mode={template_mode}...")
    refiner = ContourRefiner(
        image=image_np,
        initial_contour=contour_np,
        optimization_props=opt_props,
        sampling_props=sampl_props,
        template_props=template_props,
        template_mode=template_mode,
    )
    
    losses = []
    
    for i in tqdm(range(steps)):
        loss, metrics = refiner.step()
        losses.append(loss)
        
    return refiner, losses

In [ ]:
modes = ["fixed", "global", "per_point", "bspline", "neural"]
results = {}

for mode in modes:
    refiner, losses = run_optimization(mode, steps=150)
    results[mode] = {"refiner": refiner, "losses": losses}

In [ ]:
# Plot Losses
plt.figure(figsize=(10, 6))
for mode, res in results.items():
    plt.plot(res["losses"], label=mode)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.legend()
plt.title("Optimization Loss by Template Mode")
plt.show()

In [ ]:
# Visualize Final Contours
fig, axes = plt.subplots(1, len(modes), figsize=(4 * len(modes), 4))
if len(modes) == 1:
    axes = [axes]

for ax, mode in zip(axes, modes):
    refiner = results[mode]["refiner"]
    final_contour = refiner.get_current_contour().detach().cpu().numpy()
    
    ax.imshow(image_np, cmap="gray")
    ax.plot(gt_contour_np[:, 0], gt_contour_np[:, 1], "g--", label="GT", linewidth=1)
    ax.plot(final_contour[:, 0], final_contour[:, 1], "r-", label="Pred")
    ax.set_title(f"Mode: {mode}")
    ax.axis("off")

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Inspect Template Parameters (for modes that support it)
for mode in modes:
    refiner = results[mode]["refiner"]
    params = refiner.template_model.get_params()
    
    print(f"--- {mode} ---")
    for k, v in params.items():
        if v.numel() == 1:
            print(f"{k}: {v.item():.4f}")
        else:
            print(f"{k}: mean={v.mean().item():.4f}, std={v.std().item():.4f}, min={v.min().item():.4f}, max={v.max().item():.4f}")
    print()